# Layer 1 Output — Exploratory Data Analysis
### Intelligent AML — HT-GNN Thesis Pipeline

This notebook profiles the **19 datasets** produced by the Layer 1 ingestion run
(6,709.40 MB total) before they move into Layer 2 feature engineering / graph
construction. It is built on **DuckDB** (queries parquet directly, no need to
load multi-GB files into RAM) with **pandas/matplotlib/seaborn** for the final
aggregated tables and plots — the same stack already used elsewhere in the
project.

**What this notebook covers**

1. File inventory reconciliation against the run report
2. Structural profiling — schema, row counts, nulls, duplicates
3. Graph structure analysis (node/edge counts, density, degree distribution)
4. Temporal & burstiness analysis → motivates **Burst-Aware Temporal Decay**
5. Class imbalance analysis → motivates **GraphSMOTE / Wasserstein-GraphGAN**
6. Feature distributions, missingness, correlation
7. Flagship dataset deep dives (Elliptic v1 vs v2, PaySim vs Extended, IBM AMLSim typologies)
8. Cross-dataset master comparison table
9. Auto-generated findings tied back to the thesis's core innovations
10. Exported figures/tables for direct reuse in the thesis document

> **Before running:** edit `ROOT_DIR` in the Configuration cell below to point
> at the folder that directly contains `elliptic_v1/`, `paysim1/`, etc.
> Column-name detection is heuristic (regex over common AML/graph naming
> conventions) — use the `show_schema()` utility to sanity-check any dataset
> whose columns don't follow the usual convention.


## 0. Setup

In [ ]:
# Run once if these are not already installed in your environment
# !pip install -q duckdb pandas numpy matplotlib seaborn pyarrow openpyxl xlsxwriter


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import shutil
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["figure.autolayout"] = True

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

con = duckdb.connect(database=":memory:")
con.execute("PRAGMA threads=4;")

print("Environment ready. DuckDB version:", duckdb.__version__)


## 1. Configuration
Point `ROOT_DIR` at the folder that directly contains the dataset subfolders
shown in the Layer 1 run report (`elliptic_v1/`, `paysim1/`, `ibm_amlsim_hi_small/`, ...).

In [ ]:
# Probing candidate Layer 1 output locations dynamically
CANDIDATE_OUTPUT_DIRS = [
    Path("./layer1_output"),
    Path("./data/outputs/layer1_output"),
    Path("./graph_data"),
    Path("./data/outputs/graph_data"),
    Path("./data/processed"),
    Path("../data/processed")
]

ROOT_DIR = None
for candidate in CANDIDATE_OUTPUT_DIRS:
    if candidate.exists() and any(candidate.iterdir()):
        ROOT_DIR = candidate
        break

if ROOT_DIR is None:
    ROOT_DIR = Path("./layer1_output")
    ROOT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"⚠️ Warning: No pre-existing Layer 1 output folder found. Fallback set to: {ROOT_DIR.resolve()}")
else:
    print(f"✅ Active Layer 1 output directory: {ROOT_DIR.resolve()}")

REPORT_DIR = Path("./eda_report_artifacts")
FIG_DIR = REPORT_DIR / "figures"
TABLE_DIR = REPORT_DIR / "tables"
for d in (REPORT_DIR, FIG_DIR, TABLE_DIR):
    d.mkdir(parents=True, exist_ok=True)

SKIP_TOKENS = ("_checkpoints", "streaming")

def save_fig(name):
    plt.savefig(FIG_DIR / name, bbox_inches="tight")
    docs_fig = Path("./docs/figures")
    if docs_fig.exists():
        plt.savefig(docs_fig / name, bbox_inches="tight")

print(f"Report output directory: {REPORT_DIR.resolve()}")


## 2. Dataset Inventory (from the Layer 1 run report)

Hardcoded from the ingestion run report (19/19 datasets, 6,709.40 MB). The next
cell reconciles this against what is actually present on disk, so a silently
missing or renamed file is caught immediately rather than surfacing later as a
confusing downstream error.

In [ ]:
reported_inventory = [
    ("graph_data",                     "_checkpoints.parquet",               0.00, "system"),
    ("ulb_credit_card",                "raw_table.parquet",                 59.00, "raw_table"),
    ("data_generator",                 "edges.parquet",                      7.72, "edges"),
    ("data_generator",                 "nodes.parquet",                     27.58, "nodes"),
    ("eth_phishing",                   "edges.parquet",                    145.26, "edges"),
    ("eth_phishing",                   "nodes.parquet",                     63.95, "nodes"),
    ("ibm_amlsim_li_medium",           "edges.parquet",                    556.59, "edges"),
    ("ibm_amlsim_li_medium",           "nodes.parquet",                      9.13, "nodes"),
    ("ibm_amlsim_li_medium",           "patterns.parquet",                   0.11, "patterns"),
    ("dgraphfin",                      "edges.parquet",                     35.17, "edges"),
    ("dgraphfin",                      "nodes.parquet",                     71.95, "nodes"),
    ("cc_transactions",                "edges.parquet",                    189.20, "edges"),
    ("cc_transactions",                "nodes.parquet",                      1.04, "nodes"),
    ("xblock_eth",                     "edges.parquet",                    254.15, "edges"),
    ("xblock_eth",                     "nodes.parquet",                      8.77, "nodes"),
    ("elliptic_v2",                    "background_edges_topology.parquet",1243.78, "background_edges"),
    ("elliptic_v2",                    "background_nodes.parquet",         853.82, "background_nodes"),
    ("elliptic_v2",                    "connected_components.parquet",       0.15, "components"),
    ("elliptic_v2",                    "edges.parquet",                      4.35, "edges"),
    ("elliptic_v2",                    "nodes.parquet",                      3.09, "nodes"),
    ("ibm_amlsim_li_medium_accounts",  "raw_table.parquet",                 38.40, "raw_table"),
    ("paysim1",                        "edges.parquet",                    177.30, "edges"),
    ("paysim1",                        "nodes.parquet",                     51.10, "nodes"),
    ("eth_phishing_2nd",               "labeled_transactions.parquet",     366.44, "labeled_transactions"),
    ("smart_ponzi",                    "raw_table.parquet",                  0.10, "raw_table"),
    ("ibm_amlsim_hi_medium_accounts",  "raw_table.parquet",                 39.84, "raw_table"),
    ("ibm_amlsim_hi_medium",           "edges.parquet",                    571.62, "edges"),
    ("ibm_amlsim_hi_medium",           "nodes.parquet",                      9.33, "nodes"),
    ("ibm_amlsim_hi_medium",           "patterns.parquet",                   0.60, "patterns"),
    ("streaming",                      "part-tick-0.parquet",                0.00, "stream"),
    ("streaming",                      "part-tick-1.parquet",                0.00, "stream"),
    ("streaming",                      "part-tick-2.parquet",                0.00, "stream"),
    ("streaming",                      "part-tick-3.parquet",                0.00, "stream"),
    ("streaming",                      "part-tick-4.parquet",                0.00, "stream"),
    ("synthaml",                       "raw_table.parquet",                  0.04, "raw_table"),
    ("ibm_amlsim_hi_small_accounts",   "raw_table.parquet",                  8.77, "raw_table"),
    ("paysim_extended",                "edges.parquet",                   1306.65, "edges"),
    ("paysim_extended",                "nodes.parquet",                      1.57, "nodes"),
    ("ibm_amlsim_li_small",            "edges.parquet",                    122.49, "edges"),
    ("ibm_amlsim_li_small",            "nodes.parquet",                      3.08, "nodes"),
    ("ibm_amlsim_li_small",            "patterns.parquet",                   0.03, "patterns"),
    ("mtgox_leaked",                   "edges.parquet",                    146.73, "edges"),
    ("mtgox_leaked",                   "nodes.parquet",                      0.40, "nodes"),
    ("elliptic_v1",                    "edges.parquet",                      2.04, "edges"),
    ("elliptic_v1",                    "nodes.parquet",                     72.17, "nodes"),
    ("ibm_amlsim_li_small_accounts",   "raw_table.parquet",                 12.50, "raw_table"),
    ("ibm_amlsim_hi_small",            "edges.parquet",                     88.55, "edges"),
    ("ibm_amlsim_hi_small",            "nodes.parquet",                      2.20, "nodes"),
    ("ibm_amlsim_hi_small",            "patterns.parquet",                   0.09, "patterns"),
    ("saml_d",                         "edges.parquet",                    147.70, "edges"),
    ("saml_d",                         "nodes.parquet",                      4.84, "nodes"),
]

inv_df = pd.DataFrame(reported_inventory, columns=["dataset", "file", "size_mb_reported", "role"])
print(f"Expected files (run report): {len(inv_df)}")
print(f"Expected total size:         {inv_df['size_mb_reported'].sum():,.2f} MB")
inv_df.sort_values("size_mb_reported", ascending=False).head(10)


In [ ]:
def reconcile_with_disk(inv_df, root):
    rows = []
    for _, r in inv_df.iterrows():
        p = root / r["dataset"] / r["file"]
        exists = p.exists()
        size_actual = round(p.stat().st_size / (1024 ** 2), 2) if exists else np.nan
        rows.append({**r.to_dict(), "path": str(p), "exists_on_disk": exists, "size_mb_actual": size_actual})
    out = pd.DataFrame(rows)
    out["size_delta_mb"] = (out["size_mb_actual"] - out["size_mb_reported"]).round(2)
    return out

inv_check = reconcile_with_disk(inv_df, ROOT_DIR)
missing = inv_check[~inv_check["exists_on_disk"]]

print(f"Files found on disk: {inv_check['exists_on_disk'].sum()} / {len(inv_check)}")
if len(missing):
    print("\nMissing on disk (check ROOT_DIR, or re-run Layer 1 for these):")
    display(missing[["dataset", "file"]])
else:
    print("All files from the run report are present on disk.")

inv_check.to_csv(TABLE_DIR / "00_file_inventory_reconciled.csv", index=False)
inv_check.head(10)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

top15 = inv_df.groupby("dataset")["size_mb_reported"].sum().sort_values(ascending=False).head(15)
sns.barplot(x=top15.values, y=top15.index, ax=axes[0], palette="mako")
axes[0].set_xlabel("Size (MB)")
axes[0].set_title("Top 15 datasets by on-disk size")

role_totals = inv_df.groupby("role")["size_mb_reported"].sum().sort_values(ascending=False)
axes[1].pie(role_totals.values, labels=role_totals.index, autopct="%1.0f%%", startangle=90,
            wedgeprops={"edgecolor": "white"})
axes[1].set_title("Storage share by file role")

save_fig("01_inventory_overview.png")
plt.show()

n_datasets = inv_df["dataset"].nunique()
total_gb = inv_df["size_mb_reported"].sum() / 1024
print(f"Datasets ingested: {n_datasets} | Total files: {len(inv_df)} | Total volume: {total_gb:,.2f} GB")


## 3. Helper Functions
Column-role detection is heuristic — verify against your actual schemas with `show_schema()` if a dataset uses unconventional naming.

In [ ]:
LABEL_KEYWORDS = ["isfraud", "is_fraud", "fraud", "islaundering", "is_laundering", "laundering",
                  "illicit", "is_sar", "sar_flag", "label", "class", "target", "aml_flag", "suspicious"]
ID_KEYWORDS_SRC = ["source", "src", "from", "sender", "nameorig", "orig", "txid1", "account_from"]
ID_KEYWORDS_DST = ["target", "dst", "to", "receiver", "namedest", "dest", "txid2", "account_to"]
TIME_KEYWORDS = ["time", "timestamp", "date", "step", "tick", "block", "ts"]
AMOUNT_KEYWORDS = ["amount", "amt", "value", "weight", "balance"]
NUMERIC_TYPES = ("BIGINT", "INTEGER", "DOUBLE", "FLOAT", "DECIMAL", "HUGEINT", "SMALLINT", "TINYINT", "REAL")

def list_dataset_folders(root):
    folders = [p for p in root.iterdir() if p.is_dir() and not any(tok in p.name for tok in SKIP_TOKENS)]
    return sorted(folders, key=lambda p: p.name)

def list_parquet_files(folder):
    return sorted(folder.glob("*.parquet"))

def get_schema(path):
    return con.execute(f'DESCRIBE SELECT * FROM read_parquet(\'{path.as_posix()}\')').df()

def get_row_count(path):
    return int(con.execute(f'SELECT COUNT(*) AS n FROM read_parquet(\'{path.as_posix()}\')').df()["n"].iloc[0])

def match_column(columns, keywords):
    cols_lower = {c: c.lower().replace(" ", "").replace("-", "_") for c in columns}
    for kw in keywords:
        for orig, low in cols_lower.items():
            if kw in low:
                return orig
    return None

def guess_label_column(columns):
    return match_column(columns, LABEL_KEYWORDS)

def guess_id_columns(columns):
    src = match_column(columns, ID_KEYWORDS_SRC)
    dst = match_column(columns, ID_KEYWORDS_DST)
    if src is None or dst is None:
        id_like = [c for c in columns if "id" in c.lower()]
        if len(id_like) >= 2:
            src, dst = id_like[0], id_like[1]
    return src, dst

def guess_time_column(columns):
    return match_column(columns, TIME_KEYWORDS)

def guess_amount_column(columns):
    return match_column(columns, AMOUNT_KEYWORDS)

def numeric_columns(schema):
    return schema.loc[schema["column_type"].str.upper().str.startswith(NUMERIC_TYPES), "column_name"].tolist()

def null_pct_table(path, columns):
    exprs = ", ".join([f'AVG(CASE WHEN "{c}" IS NULL THEN 1.0 ELSE 0.0 END) AS "{c}"' for c in columns])
    q = f'SELECT {exprs} FROM read_parquet(\'{path.as_posix()}\')'
    res = con.execute(q).df().T
    res.columns = ["null_pct"]
    return (res["null_pct"] * 100).round(3)

def show_schema(dataset_name, file_name=None):
    folder = ROOT_DIR / dataset_name
    files = [file_name] if file_name else [f.name for f in list_parquet_files(folder)]
    for fn in files:
        print(f"\n{dataset_name}/{fn}")
        display(get_schema(folder / fn))

print("Helper functions ready.")
# Example manual check: show_schema("elliptic_v1")


## 4. Structural Profiling — Schema, Row Counts, Nulls, Duplicates

In [ ]:
profile_rows = []
schema_cache = {}  # (dataset, file) -> schema dataframe

for folder in list_dataset_folders(ROOT_DIR):
    for f in list_parquet_files(folder):
        try:
            schema = get_schema(f)
            n_rows = get_row_count(f)
            schema_cache[(folder.name, f.name)] = schema
            profile_rows.append({
                "dataset": folder.name, "file": f.name, "n_rows": n_rows, "n_cols": len(schema),
                "columns_preview": ", ".join(schema["column_name"].tolist()[:8]) + (" ..." if len(schema) > 8 else ""),
                "size_mb": round(f.stat().st_size / (1024 ** 2), 2),
            })
        except Exception as e:
            print(f"Could not profile {folder.name}/{f.name}: {e}")

profile_df = pd.DataFrame(profile_rows).sort_values(["dataset", "file"]).reset_index(drop=True)
profile_df.to_csv(TABLE_DIR / "01_structural_profile.csv", index=False)
print(f"Profiled {len(profile_df)} parquet files across {profile_df['dataset'].nunique()} datasets.")
profile_df


In [ ]:
null_summary = []

for (dataset, file), schema in schema_cache.items():
    path = ROOT_DIR / dataset / file
    cols = schema["column_name"].tolist()
    try:
        nulls = null_pct_table(path, cols)
        worst = nulls.sort_values(ascending=False).head(3)
        null_summary.append({
            "dataset": dataset, "file": file,
            "max_null_pct": round(nulls.max(), 2),
            "cols_with_nulls": int((nulls > 0).sum()),
            "worst_columns": ", ".join([f"{c} ({v:.1f}%)" for c, v in worst.items() if v > 0]) or "none",
        })
    except Exception as e:
        print(f"Null scan failed for {dataset}/{file}: {e}")

null_df = pd.DataFrame(null_summary).sort_values("max_null_pct", ascending=False)
null_df.to_csv(TABLE_DIR / "02_null_summary.csv", index=False)
null_df.head(20)


## 5. Graph Structure Analysis (edge files)
Node/edge counts, density, self-loops, and degree distribution for every `*edges*.parquet` file.

In [ ]:
graph_stats = []

for (dataset, file), schema in schema_cache.items():
    if "edge" not in file.lower():
        continue
    path = ROOT_DIR / dataset / file
    cols = schema["column_name"].tolist()
    src, dst = guess_id_columns(cols)
    if not src or not dst:
        print(f"Could not identify source/target columns for {dataset}/{file}; columns = {cols[:10]}")
        continue
    try:
        base_q = (
            f'WITH e AS (SELECT "{src}" AS s, "{dst}" AS t FROM read_parquet(\'{path.as_posix()}\')) '
            f'SELECT COUNT(*) AS n_edges, COUNT(DISTINCT s) AS n_src, COUNT(DISTINCT t) AS n_dst, '
            f'SUM(CASE WHEN s = t THEN 1 ELSE 0 END) AS self_loops FROM e'
        )
        res = con.execute(base_q).df().iloc[0]

        union_q = (
            f'SELECT COUNT(*) AS n FROM ('
            f'SELECT "{src}" AS node FROM read_parquet(\'{path.as_posix()}\') '
            f'UNION SELECT "{dst}" AS node FROM read_parquet(\'{path.as_posix()}\'))'
        )
        n_nodes_approx = int(con.execute(union_q).df()["n"].iloc[0])
        avg_degree = 2 * res["n_edges"] / n_nodes_approx if n_nodes_approx else np.nan

        graph_stats.append({
            "dataset": dataset, "file": file, "source_col": src, "target_col": dst,
            "n_edges": int(res["n_edges"]), "n_nodes_approx": n_nodes_approx,
            "avg_degree": round(avg_degree, 3),
            "self_loop_pct": round(100 * res["self_loops"] / res["n_edges"], 4) if res["n_edges"] else 0,
        })
    except Exception as e:
        print(f"Graph stats failed for {dataset}/{file}: {e}")

graph_df = pd.DataFrame(graph_stats).sort_values("n_edges", ascending=False).reset_index(drop=True)
graph_df.to_csv(TABLE_DIR / "03_graph_structure_summary.csv", index=False)
graph_df


In [ ]:
FEATURED_GRAPHS = graph_df.head(6)[["dataset", "file", "source_col"]].values.tolist()
n_feat = len(FEATURED_GRAPHS)

if n_feat:
    ncols = 3
    nrows = int(np.ceil(n_feat / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4.5 * nrows))
    axes = np.array(axes).flatten()

    for ax, (dataset, file, src) in zip(axes, FEATURED_GRAPHS):
        path = ROOT_DIR / dataset / file
        q = f'SELECT "{src}" AS node, COUNT(*) AS degree FROM read_parquet(\'{path.as_posix()}\') GROUP BY "{src}"'
        deg = con.execute(q).df()
        ax.hist(deg["degree"], bins=40, log=True, color="#4C72B0", edgecolor="white")
        ax.set_xscale("log")
        ax.set_title(dataset, fontsize=10)
        ax.set_xlabel("out-degree (log)")
        ax.set_ylabel("node count (log)")

    for ax in axes[n_feat:]:
        ax.axis("off")

    plt.suptitle("Out-degree distributions - largest edge sets", y=1.02, fontsize=13)
    save_fig("02_degree_distributions.png")
    plt.show()
else:
    print("No edge files with detectable source/target columns.")


## 6. Temporal & Burst Pattern Analysis
Directly relevant to **Burst-Aware Temporal Decay**: datasets with high burstiness (large coefficient of variation across time buckets) are exactly where a fixed-window temporal encoder underfits.

In [ ]:
burst_stats = []

for (dataset, file), schema in schema_cache.items():
    fl = file.lower()
    if not any(tok in fl for tok in ("edge", "raw_table", "labeled")):
        continue
    cols = schema["column_name"].tolist()
    tcol = guess_time_column(cols)
    if not tcol:
        continue
    path = ROOT_DIR / dataset / file
    try:
        ts_type = schema.loc[schema["column_name"] == tcol, "column_type"].iloc[0].upper()
        if any(t in ts_type for t in ("INT", "DOUBLE", "DECIMAL", "REAL")):
            bucket_expr = f'CAST("{tcol}" AS BIGINT)'
        else:
            bucket_expr = f'date_trunc(\'day\', CAST("{tcol}" AS TIMESTAMP))'

        q = f'SELECT {bucket_expr} AS bucket, COUNT(*) AS n FROM read_parquet(\'{path.as_posix()}\') GROUP BY 1 ORDER BY 1'
        ts = con.execute(q).df()
        if len(ts) < 3:
            continue

        mean_n, std_n = ts["n"].mean(), ts["n"].std()
        burstiness = std_n / mean_n if mean_n else np.nan

        burst_stats.append({
            "dataset": dataset, "file": file, "time_col": tcol, "n_buckets": len(ts),
            "mean_per_bucket": round(mean_n, 2), "burstiness_cv": round(burstiness, 3),
            "max_bucket_count": int(ts["n"].max()),
        })
    except Exception as e:
        print(f"Temporal scan failed for {dataset}/{file}: {e}")

burst_df = pd.DataFrame(burst_stats).sort_values("burstiness_cv", ascending=False).reset_index(drop=True)
burst_df.to_csv(TABLE_DIR / "04_temporal_burstiness.csv", index=False)
burst_df


In [ ]:
TOP_BURSTY = burst_df.head(4)
n_top = len(TOP_BURSTY)

if n_top:
    fig, axes = plt.subplots(n_top, 1, figsize=(13, 3 * n_top))
    axes = [axes] if n_top == 1 else list(axes)

    for ax, (_, row) in zip(axes, TOP_BURSTY.iterrows()):
        path = ROOT_DIR / row["dataset"] / row["file"]
        tcol = row["time_col"]
        schema = schema_cache[(row["dataset"], row["file"])]
        ts_type = schema.loc[schema["column_name"] == tcol, "column_type"].iloc[0].upper()
        bucket_expr = (f'CAST("{tcol}" AS BIGINT)' if any(t in ts_type for t in ("INT", "DOUBLE", "DECIMAL", "REAL"))
                       else f'date_trunc(\'day\', CAST("{tcol}" AS TIMESTAMP))')
        q = f'SELECT {bucket_expr} AS bucket, COUNT(*) AS n FROM read_parquet(\'{path.as_posix()}\') GROUP BY 1 ORDER BY 1'
        ts = con.execute(q).df()
        ax.plot(ts["bucket"], ts["n"], color="#C44E52", linewidth=1)
        ax.fill_between(ts["bucket"], ts["n"], alpha=0.15, color="#C44E52")
        ax.set_title(f"{row['dataset']} - burstiness CV = {row['burstiness_cv']:.2f}", fontsize=10)
        ax.set_ylabel("events / bucket")

    plt.suptitle("Most bursty transaction time series", y=1.01)
    save_fig("03_burst_time_series.png")
    plt.show()
else:
    print("No datasets with a detectable time column.")


## 7. Class Imbalance & Label Distribution
Directly relevant to **GraphSMOTE / Wasserstein-GraphGAN**: severity here is the empirical justification for synthetic minority oversampling over plain class weighting.

In [ ]:
imbalance_stats = []

for (dataset, file), schema in schema_cache.items():
    cols = schema["column_name"].tolist()
    lcol = guess_label_column(cols)
    if not lcol:
        continue
    path = ROOT_DIR / dataset / file
    try:
        q = f'SELECT "{lcol}" AS label, COUNT(*) AS n FROM read_parquet(\'{path.as_posix()}\') GROUP BY 1 ORDER BY n DESC'
        vc = con.execute(q).df()
        if len(vc) < 2:
            continue
        majority, minority = vc["n"].iloc[0], vc["n"].iloc[-1]
        ratio = majority / minority if minority else np.inf
        imbalance_stats.append({
            "dataset": dataset, "file": file, "label_col": lcol, "n_classes": len(vc),
            "majority_n": int(majority), "minority_n": int(minority),
            "imbalance_ratio": ratio,
        })
    except Exception as e:
        print(f"Label scan failed for {dataset}/{file}: {e}")

imbalance_df = pd.DataFrame(imbalance_stats).sort_values("imbalance_ratio", ascending=False).reset_index(drop=True)
imbalance_df.to_csv(TABLE_DIR / "05_class_imbalance.csv", index=False)
imbalance_df


In [ ]:
plot_df = imbalance_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["imbalance_ratio"]).copy()
plot_df = plot_df.sort_values("imbalance_ratio", ascending=True)

if len(plot_df):
    plt.figure(figsize=(9, max(3, 0.35 * len(plot_df))))
    sns.barplot(data=plot_df, x="imbalance_ratio", y="dataset", palette="rocket")
    plt.xscale("log")
    plt.xlabel("Majority : Minority ratio (log scale)")
    plt.title("Label imbalance severity across datasets")
    plt.axvline(50, color="black", linestyle="--", linewidth=1, label="50:1 severe-imbalance threshold")
    plt.legend()
    save_fig("04_class_imbalance.png")
    plt.show()

    severe = plot_df[plot_df["imbalance_ratio"] > 50]
    print(f"{len(severe)} / {len(plot_df)} labeled datasets exceed a 50:1 imbalance ratio.")
else:
    severe = plot_df
    print("No datasets with a detectable multi-class label column.")


## 8. Feature Distributions, Missingness & Correlation
Numeric feature correlation for node/account-level files, sampled to keep this fast on multi-GB inputs.

In [ ]:
FEATURE_TARGETS = [(d, f) for (d, f) in schema_cache.keys() if "node" in f.lower() or "raw_table" in f.lower()]

for dataset, file in FEATURE_TARGETS[:8]:
    schema = schema_cache[(dataset, file)]
    path = ROOT_DIR / dataset / file
    num_cols = numeric_columns(schema)
    num_cols = [c for c in num_cols if c.lower() not in ("txid", "id", "node_id", "account_id")][:12]
    if len(num_cols) < 2:
        continue
    try:
        cols_sql = ", ".join([f'"{c}"' for c in num_cols])
        q = f'SELECT {cols_sql} FROM read_parquet(\'{path.as_posix()}\') USING SAMPLE 100000 ROWS'
        sample = con.execute(q).df()
        corr = sample.corr(numeric_only=True)

        fig, ax = plt.subplots(figsize=(min(1 + 0.6 * len(num_cols), 10), min(1 + 0.6 * len(num_cols), 8)))
        sns.heatmap(corr, cmap="vlag", center=0, annot=len(num_cols) <= 10, fmt=".2f",
                    square=True, cbar_kws={"shrink": .7}, ax=ax)
        ax.set_title(f"{dataset}/{file} - numeric feature correlation")
        save_fig(f"05_corr_{dataset}.png")
        plt.show()
    except Exception as e:
        print(f"Correlation failed for {dataset}/{file}: {e}")


## 9. Flagship Dataset Deep Dives

In [ ]:
print("=== Elliptic v1 vs v2 ===")
for version in ["elliptic_v1", "elliptic_v2"]:
    node_path = ROOT_DIR / version / "nodes.parquet"
    if not node_path.exists():
        continue
    schema = get_schema(node_path)
    cols = schema["column_name"].tolist()
    lcol = guess_label_column(cols)
    n_total = get_row_count(node_path)
    print(f"\n{version}: {n_total:,} nodes | label column guessed = {lcol}")
    if lcol:
        q = f'SELECT "{lcol}" AS label, COUNT(*) AS n FROM read_parquet(\'{node_path.as_posix()}\') GROUP BY 1 ORDER BY n DESC'
        display(con.execute(q).df())

bg_path = ROOT_DIR / "elliptic_v2" / "background_nodes.parquet"
lab_path = ROOT_DIR / "elliptic_v2" / "nodes.parquet"
if bg_path.exists() and lab_path.exists():
    bg_n, lab_n = get_row_count(bg_path), get_row_count(lab_path)
    print(f"\nelliptic_v2 background graph: {bg_n:,} nodes vs labeled subgraph: {lab_n:,} nodes "
          f"({100 * lab_n / bg_n:.3f}% labeled) - illustrates the semi-supervised scale gap.")


In [ ]:
print("=== PaySim vs PaySim Extended ===")
for name in ["paysim1", "paysim_extended"]:
    path = ROOT_DIR / name / "edges.parquet"
    if not path.exists():
        continue
    schema = get_schema(path)
    cols = schema["column_name"].tolist()
    lcol = guess_label_column(cols)
    amt_col = guess_amount_column(cols)
    n = get_row_count(path)
    print(f"\n{name}: {n:,} transactions")
    if lcol:
        q = f'SELECT AVG(CASE WHEN CAST("{lcol}" AS DOUBLE) > 0 THEN 1.0 ELSE 0.0 END) * 100 AS fraud_pct FROM read_parquet(\'{path.as_posix()}\')'
        try:
            rate = con.execute(q).df()
            print(f"  Fraud rate: {rate['fraud_pct'].iloc[0]:.4f}%")
        except Exception as e:
            print(f"  Could not compute fraud rate: {e}")
    if amt_col:
        q = f'SELECT MIN("{amt_col}") AS min_amt, AVG("{amt_col}") AS avg_amt, MAX("{amt_col}") AS max_amt FROM read_parquet(\'{path.as_posix()}\')'
        display(con.execute(q).df())


In [ ]:
print("=== IBM AMLSim laundering typology breakdown ===")
typology_frames = {}

for name in ["ibm_amlsim_hi_small", "ibm_amlsim_li_small", "ibm_amlsim_hi_medium", "ibm_amlsim_li_medium"]:
    ppath = ROOT_DIR / name / "patterns.parquet"
    if not ppath.exists():
        continue
    schema = get_schema(ppath)
    cols = schema["column_name"].tolist()
    type_col = match_column(cols, ["type", "pattern", "typology", "scenario"])
    if not type_col:
        print(f"{name}: pattern-type column not identified, columns = {cols}")
        continue
    q = f'SELECT "{type_col}" AS pattern_type, COUNT(*) AS n FROM read_parquet(\'{ppath.as_posix()}\') GROUP BY 1 ORDER BY n DESC'
    vc = con.execute(q).df()
    typology_frames[name] = vc
    print(f"\n{name}:")
    display(vc)

if typology_frames:
    combined = pd.concat(
        [df.assign(dataset=name) for name, df in typology_frames.items()], ignore_index=True
    )
    pivot = combined.pivot_table(index="pattern_type", columns="dataset", values="n", fill_value=0)
    pivot.plot(kind="bar", figsize=(11, 5), colormap="tab10")
    plt.title("Laundering typology counts across IBM AMLSim variants")
    plt.ylabel("count")
    plt.xticks(rotation=30, ha="right")
    save_fig("06_ibm_amlsim_typologies.png")
    plt.show()


In [ ]:
print("=== Quick profiles: DGraphFin, ULB Credit Card, SAML-D ===")
for name, fname in [("dgraphfin", "nodes.parquet"), ("ulb_credit_card", "raw_table.parquet"), ("saml_d", "edges.parquet")]:
    path = ROOT_DIR / name / fname
    if not path.exists():
        continue
    schema = get_schema(path)
    cols = schema["column_name"].tolist()
    lcol = guess_label_column(cols)
    n = get_row_count(path)
    print(f"\n{name}/{fname}: {n:,} rows, {len(cols)} cols, label column guessed = {lcol}")
    if lcol:
        q = f'SELECT "{lcol}" AS label, COUNT(*) AS n FROM read_parquet(\'{path.as_posix()}\') GROUP BY 1 ORDER BY n DESC LIMIT 10'
        display(con.execute(q).df())


## 10. Cross-Dataset Master Comparison Table

In [ ]:
master = (
    profile_df.groupby("dataset")
    .agg(n_files=("file", "count"), total_rows=("n_rows", "sum"), total_size_mb=("size_mb", "sum"))
    .reset_index()
)

merge_specs = [
    (graph_df, ["dataset", "n_edges", "n_nodes_approx", "avg_degree", "self_loop_pct"]),
    (burst_df, ["dataset", "burstiness_cv"]),
    (imbalance_df, ["dataset", "imbalance_ratio", "n_classes"]),
    (null_df, ["dataset", "max_null_pct"]),
]

for df_, cols_ in merge_specs:
    if len(df_) == 0:
        continue
    agg = df_[cols_].groupby("dataset").first().reset_index()
    master = master.merge(agg, on="dataset", how="left")

master = master.sort_values("total_size_mb", ascending=False).reset_index(drop=True)
master.to_csv(TABLE_DIR / "06_master_comparison.csv", index=False)
try:
    master.to_excel(TABLE_DIR / "06_master_comparison.xlsx", index=False)
except Exception as e:
    print(f"xlsx export skipped: {e}")
master


## 11. Key Findings & Modeling Recommendations

In [ ]:
findings = []

if len(plot_df):
    worst_imb = plot_df.iloc[-1]
    n_severe = len(plot_df[plot_df["imbalance_ratio"] > 50])
    findings.append(
        f"- {n_severe} / {len(plot_df)} labeled datasets exceed a 50:1 class-imbalance ratio "
        f"(worst: {worst_imb['dataset']} at {worst_imb['imbalance_ratio']:.0f}:1). "
        f"Supports GraphSMOTE / Wasserstein-GraphGAN oversampling over plain class weighting for these."
    )

if len(burst_df):
    top_burst = burst_df.iloc[0]
    findings.append(
        f"- {top_burst['dataset']} shows the highest burstiness (CV = {top_burst['burstiness_cv']:.2f}), "
        f"supporting Burst-Aware Temporal Decay over a fixed-window temporal encoder."
    )

if len(graph_df):
    by_degree = graph_df.sort_values("avg_degree")
    sparse, dense = by_degree.iloc[0], by_degree.iloc[-1]
    findings.append(
        f"- Graph density varies widely: {sparse['dataset']} averages {sparse['avg_degree']:.2f} degree "
        f"vs {dense['dataset']} at {dense['avg_degree']:.2f}. Tune HGTConv neighbor-sampling fan-out per dataset."
    )

if len(null_df) and null_df["max_null_pct"].max() > 20:
    worst_null = null_df.iloc[0]
    findings.append(
        f"- {worst_null['dataset']}/{worst_null['file']} has up to {worst_null['max_null_pct']:.1f}% "
        f"missingness in at least one column - impute or drop before feature encoding."
    )

print("\n".join(findings) if findings else "No automatic flags raised - see the tables above for full detail.")


**Manual notes to fold into the thesis write-up:**

- Per-dataset imbalance ratios (Section 7) are the empirical anchor for the
  Wasserstein-GraphGAN + GraphSMOTE ablation table — cite the exact ratios
  rather than a generic "severe imbalance" claim.
- Burstiness ranking (Section 6) gives a concrete ordering of which datasets
  should show the largest AUC/F1 gain from Burst-Aware Temporal Decay vs a
  vanilla TGN/EvolveGCN baseline — useful for the ablation study design.
- The Elliptic v1 → v2 labeled-fraction comparison (Section 9) is a clean
  illustration of the semi-supervised setting that motivates Task-Free
  Continual Graph Learning over a static train/test split.
- IBM AMLSim typology counts (Section 9) map directly onto the circular
  laundering worked example in the diagram suite — use the pivot table there
  to pick a representative fan-out/fan-in/cycle case with enough support.


## 12. Export Artifacts

In [ ]:
manifest = []
for p in sorted(FIG_DIR.glob("*.png")) + sorted(TABLE_DIR.glob("*")):
    manifest.append({"artifact": p.name, "path": str(p), "kind": p.suffix.lstrip(".")})
manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(REPORT_DIR / "manifest.csv", index=False)

zip_path = shutil.make_archive(str(REPORT_DIR), "zip", root_dir=REPORT_DIR)
print(f"Saved {len(manifest_df)} artifacts to {REPORT_DIR.resolve()}")
print(f"Zipped report bundle: {zip_path}")
manifest_df
